# Personal baselines and data-quality flags
Each chart shows the value, your own 28-day baseline (median of the previous valid days) and a band of ±1.5 spread. Red crosses are flagged days (short night, HRV outlier). Thresholds are in `config/rules.yaml`. Run `scripts/backfill.py` first.

In [ ]:
import sys
from datetime import date
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "src"))
from training_agent.storage.sqlite import SqliteRepository
from training_agent.planner.baselines import assess, load_params

plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": 0.3})
repo = SqliteRepository(Path("..") / "data" / "db" / "training.sqlite")
assessments = assess(repo.get_metrics(date(2000, 1, 1), date(2100, 1, 1)), load_params())

rows = []
for a in assessments:
    r = {"date": a.date, "flags": ",".join(a.flags)}
    for sig, s in a.signals.items():
        r.update({f"{sig}": s.value, f"{sig}_base": s.baseline, f"{sig}_spread": s.spread, f"{sig}_z": s.z})
    rows.append(r)
A = pd.DataFrame(rows).set_index("date"); A.index = pd.to_datetime(A.index)
print(len(A), "days")
print("Flag counts:"); print(A["flags"].str.split(",").explode().replace("", pd.NA).value_counts().to_string())

## Value vs your personal baseline

In [ ]:
def band(sig, title):
    fig, ax = plt.subplots()
    base, sp = A[f"{sig}_base"], A[f"{sig}_spread"]
    ax.fill_between(A.index, base - 1.5*sp, base + 1.5*sp, alpha=0.2, label="normal range")
    ax.plot(A.index, base, color="gray", lw=1, label="baseline")
    ax.plot(A.index, A[sig], marker="o", ms=3, lw=1, label="value")
    flagged = A[A["flags"].str.contains("short_sleep|hrv_outlier")]
    ax.scatter(flagged.index, flagged[sig], marker="x", color="red", zorder=5, label="flagged")
    ax.set_title(title); ax.legend(ncol=4, fontsize=8); plt.show()

band("hrv_last_night_ms", "HRV vs personal baseline (ms)")
band("resting_hr", "Resting HR vs personal baseline (bpm)")
band("sleep_hours", "Sleep hours vs personal baseline")
band("body_battery_morning", "Morning Body Battery vs personal baseline")

## Latest day vs your normal
`z` is the distance from your baseline in spread units. Roughly: |z| < 1.5 normal, 1.5 to 2.5 notable, above 2.5 unusual.

In [ ]:
last = assessments[-1]
t = pd.DataFrame({s: {"value": v.value, "baseline": v.baseline, "z": v.z} for s, v in last.signals.items()}).T.round(2)
print(last.date, "flags:", last.flags or "none"); display(t)